# Site04: SPICE geometric Sun/Earth baseline

**Purpose:** Calculate and preserve a first date-specific geometric sky-position baseline at the fixed Site04 test pixel S04-P02.

**Status:** Runnable on Hub with mounted Site04 data. Console lines show UTC timestamps, elapsed seconds since setup, and step progress (`START`/`DONE`, kernel `[i/n]`, ephemeris `[done/total]`). Re-run **Run All** after updating from git; keep `SCIENCE_CALCULATIONS_READY=false` until benchmark review.

**What this notebook calculates:** the geometric centre azimuth and elevation of the Sun and Earth as seen from one LOLA DEM sample point at 24 hourly UTC epochs on 2025-01-01. It records source-kernel SHA-256 values and saves CSV/JSON results in the user's persistent home directory.

**What it does not calculate:** DEM terrain horizons, partial Sun/Earth disk visibility, solar power, communications availability, landing safety, or a site ranking. The local vertical is radial to the product's 1,737,400 m reference sphere, not a DEM-derived terrain normal. The outputs are geometric centre angles only.

The fixed point is taken from notebook 02: projected X=−2652.5 m, Y=−6082.5 m, LDEC=2, Z=1736.902 m. NASA's PGDA README says the products use pixel-registered X/Y and Z is surface height; Barker et al. describes the product height as elevation above the reference sphere. This notebook uses radius = 1,737,400 m + Z. It uses the product's stated MOON_ME/DE421 frame and does not add a PA↔ME rotation.

In [ ]:
from __future__ import annotations

from datetime import datetime, timedelta, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import time
import urllib.request

import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS, Transformer
import spiceypy as spice
from IPython.display import display

_DATA_DIR_CANDIDATES = (
    Path("/home/jovyan/data/LOLA_5mpp/Site04"),
    Path("/workspace/data/LOLA_5mpp/Site04"),
)
DATA_DIR = next((p for p in _DATA_DIR_CANDIDATES if p.is_dir()), _DATA_DIR_CANDIDATES[0])
SURF_PATH = DATA_DIR / "Site04_final_adj_5mpp_surf.tif"
LDEC_PATH = DATA_DIR / "Site04_final_adj_5mpp_ldec.tif"

POINT_ID = "S04-P02"
X_M = -2652.5
Y_M = -6082.5
EXPECTED_ROW = 1416
EXPECTED_COL = 1269
REFERENCE_RADIUS_M = 1_737_400.0
EXPECTED_Z_M = 1736.902
EXPECTED_LDEC = 2

OUTPUT_DIR = Path.home() / "research-results" / "site04-spice-baseline"
KERNEL_DIR = Path.home() / ".cache" / "lunar-navigator" / "de421-kernels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
KERNEL_DIR.mkdir(parents=True, exist_ok=True)

NOTEBOOK_RUN_T0 = time.perf_counter()


def log(msg: str) -> None:
    elapsed = time.perf_counter() - NOTEBOOK_RUN_T0
    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    print(f"[{stamp}] (+{elapsed:6.1f}s) {msg}", flush=True)


class run_step:
    """Context manager: START/DONE lines with wall-clock duration."""

    def __init__(self, label: str):
        self.label = label

    def __enter__(self):
        self._t0 = time.perf_counter()
        log(f"START {self.label}")
        return self

    def __exit__(self, exc_type, exc, _tb):
        dt = time.perf_counter() - self._t0
        if exc_type is not None:
            log(f"FAIL  {self.label} ({dt:.2f}s) — {exc_type.__name__}: {exc}")
        else:
            log(f"DONE  {self.label} ({dt:.2f}s)")
        return False


def utc_to_et(epoch_utc: datetime) -> float:
    """NAIF calendar UTC string (avoid ISO-T, which str2et rejects in CSPICE N0067)."""
    cal = epoch_utc.strftime("%Y %b %d %H:%M:%S UTC").upper()
    return float(spice.str2et(cal))


log("notebook setup")
print("data_dir_exists:", DATA_DIR.is_dir())
print("output_dir:", OUTPUT_DIR)
print("kernel_dir:", KERNEL_DIR)
if not SURF_PATH.is_file() or not LDEC_PATH.is_file():
    raise FileNotFoundError(
        f"Expected the Site04 products in {DATA_DIR}. "
        "Mount LOLA under /home/jovyan/data (Hub) or /workspace/data (research profile)."
    )

## 1. Download and fingerprint the DE421 kernel set

The notebook downloads only from NAIF's HTTPS server into the user's cache. SHA-256 values are computed locally and written to the run manifest so the exact inputs can be reproduced. A locally computed checksum identifies the downloaded file; it is not a separate signature or proof of publisher authenticity. The SPICE calls below also serve as a practical coverage check: a missing ephemeris segment or frame definition will raise an error.

In [ ]:
KERNELS = {
    "naif0012.tls": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/lsk/naif0012.tls",
    "de421.bsp": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/spk/planets/a_old_versions/de421.bsp",
    "moon_pa_de421_1900-2050.bpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/moon_pa_de421_1900-2050.bpc",
    "moon_080317.tf": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/fk/satellites/moon_080317.tf",
    "pck00011.tpc": "https://naif.jpl.nasa.gov/pub/naif/generic_kernels/pck/pck00011.tpc",
}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def download_if_missing(filename: str, url: str) -> Path:
    destination = KERNEL_DIR / filename
    if destination.is_file() and destination.stat().st_size > 0:
        return destination
    temporary = destination.with_suffix(destination.suffix + ".part")
    request = urllib.request.Request(url, headers={"User-Agent": "lunar-navigator-research/1.0"})
    try:
        with urllib.request.urlopen(request, timeout=90) as response, temporary.open("wb") as out:
            while True:
                chunk = response.read(1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)
        if temporary.stat().st_size < 100:
            raise IOError(f"Downloaded file is unexpectedly small: {filename}")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination

with run_step("download and fingerprint kernels"):
    kernel_paths = {}
    for i, (name, url) in enumerate(KERNELS.items(), 1):
        log(f"  kernel [{i}/{len(KERNELS)}] {name}")
        kernel_paths[name] = download_if_missing(name, url)
    kernel_manifest = [
        {
            "filename": name,
            "url": KERNELS[name],
            "bytes": path.stat().st_size,
            "sha256": sha256_file(path),
        }
        for name, path in kernel_paths.items()
    ]
display(pd.DataFrame(kernel_manifest))

In [ ]:
with run_step("load SPICE kernels into session"):
    spice.kclear()
    for i, name in enumerate(KERNELS, 1):
        log(f"  furnsh [{i}/{len(KERNELS)}] {name}")
        spice.furnsh(str(kernel_paths[name]))
    frame_id = spice.namfrm("MOON_ME")
    if frame_id == 0:
        raise RuntimeError("MOON_ME frame did not resolve; inspect the loaded lunar FK/PCK kernels.")
    print("MOON_ME frame ID:", frame_id)
    print("Loaded kernel count:", spice.ktotal("ALL"))

## 2. Sample the fixed pixel and validate the inverse projection

The product README specifies pixel-registered coordinates (pixel centres). We verify that the fixed projected coordinate maps back to the expected centre and that the DEM values match the audited sample. The inverse/forward round trip checks the projection implementation; it does not independently prove the frame association, which comes from the product documentation and paper.

In [ ]:
with run_step("sample DEM pixel and projection round-trip"):
    with rasterio.open(SURF_PATH) as surf, rasterio.open(LDEC_PATH) as ldec:
        if surf.crs is None or ldec.crs is None:
            raise ValueError("Both rasters must have a CRS.")
        if surf.transform != ldec.transform or surf.width != ldec.width or surf.height != ldec.height:
            raise ValueError("Surface and LDEC rasters do not share the audited grid.")

        row, col = surf.index(X_M, Y_M)
        center_x, center_y = surf.xy(row, col, offset="center")
        z_sample = float(surf.read(1, masked=True)[row, col])
        ldec_sample = int(ldec.read(1, masked=True)[row, col])

        source_crs = CRS.from_wkt(surf.crs.to_wkt())
        geographic_crs = source_crs.geodetic_crs
        if geographic_crs is None:
            raise ValueError("Could not derive the raster's geographic CRS.")
        inverse = Transformer.from_crs(source_crs, geographic_crs, always_xy=True)
        forward = Transformer.from_crs(geographic_crs, source_crs, always_xy=True)
        longitude_deg, latitude_deg = inverse.transform(X_M, Y_M)
        x_roundtrip, y_roundtrip = forward.transform(longitude_deg, latitude_deg)

    # Guard against accidentally sampling a different pixel or changed local dataset.
    assert (row, col) == (EXPECTED_ROW, EXPECTED_COL), (row, col)
    assert math.isclose(center_x, X_M, abs_tol=1e-8)
    assert math.isclose(center_y, Y_M, abs_tol=1e-8)
    assert math.isclose(z_sample, EXPECTED_Z_M, abs_tol=0.01), z_sample
    assert ldec_sample == EXPECTED_LDEC, ldec_sample
    assert math.hypot(x_roundtrip - X_M, y_roundtrip - Y_M) < 1e-3
    assert -90.0 <= latitude_deg < -89.0, latitude_deg

    ellipsoid = source_crs.ellipsoid
    print({
        "point_id": POINT_ID,
        "row": row,
        "col": col,
        "center_x_m": center_x,
        "center_y_m": center_y,
        "z_m_above_reference_sphere": z_sample,
        "ldec": ldec_sample,
        "longitude_deg_from_raster_crs": longitude_deg,
        "latitude_deg_from_raster_crs": latitude_deg,
        "projection_roundtrip_residual_m": math.hypot(x_roundtrip-X_M, y_roundtrip-Y_M),
        "crs_semi_major_m": ellipsoid.semi_major_metre,
        "crs_inverse_flattening": ellipsoid.inverse_flattening,
        "crs_wkt": source_crs.to_wkt(),
    })

## 3. Build the observer and compute the hourly series

The DEM Z value is elevation above the reference sphere, so the observer's spherical radius is reference radius + Z. The raster's inverse projection supplies longitude/latitude on that sphere. We express the point in the product's MOON_ME axes using SPICE's spherical-to-rectangular conversion.

For each UTC epoch, SPICE returns geometric position vectors in kilometres; the sampled DEM point is converted from metres to kilometres before subtraction. The target position is relative to the Moon centre, expressed in MOON_ME. Subtracting the fixed observer vector gives the site-to-target vector. The local basis uses radial up and the usual increasing-latitude north/east directions. Azimuth is clockwise from north, in [0, 360); elevation is measured from the local radial tangent plane.

No light-time correction is applied (`NONE`), and target disc size is not modeled. Do not interpret positive elevation as terrain-clear visibility.

In [ ]:
with run_step("build observer frame and local basis"):
    observer_radius_m = REFERENCE_RADIUS_M + z_sample
    observer_radius_km = observer_radius_m / 1000.0
    observer_xyz_km = np.asarray(
        spice.latrec(
            observer_radius_km,
            math.radians(longitude_deg),
            math.radians(latitude_deg),
        ),
        dtype=float,
    )
    lon = math.radians(longitude_deg)
    lat = math.radians(latitude_deg)
    east_hat = np.array([-math.sin(lon), math.cos(lon), 0.0])
    north_hat = np.array([
        -math.sin(lat) * math.cos(lon),
        -math.sin(lat) * math.sin(lon),
        math.cos(lat),
    ])
    up_hat = observer_xyz_km / np.linalg.norm(observer_xyz_km)
    assert np.allclose(np.linalg.norm(east_hat), 1.0, atol=1e-12)
    assert np.allclose(np.linalg.norm(north_hat), 1.0, atol=1e-12)
    assert abs(float(np.dot(east_hat, north_hat))) < 1e-12
    assert abs(float(np.dot(east_hat, up_hat))) < 1e-3
    assert abs(float(np.dot(north_hat, up_hat))) < 1e-3
    log(f"  observer radius {observer_radius_m:.3f} m; |r|={np.linalg.norm(observer_xyz_km):.6f} km")

def local_angles(vector_xyz):
    east = float(np.dot(vector_xyz, east_hat))
    north = float(np.dot(vector_xyz, north_hat))
    up = float(np.dot(vector_xyz, up_hat))
    azimuth_deg = math.degrees(math.atan2(east, north)) % 360.0
    elevation_deg = math.degrees(math.atan2(up, math.hypot(east, north)))
    return azimuth_deg, elevation_deg

start_utc = datetime(2025, 1, 1, 0, 0, 0, tzinfo=timezone.utc)
epochs = [start_utc + timedelta(hours=i) for i in range(24)]
records = []
n_epochs = len(epochs)
n_targets = 2
n_total = n_epochs * n_targets

with run_step(f"hourly ephemeris ({n_epochs} UTC epochs × Sun/Earth)"):
    for hour_idx, epoch_utc in enumerate(epochs, 1):
        et = utc_to_et(epoch_utc)
        for target in ("SUN", "EARTH"):
            target_from_moon, _light_time_s = spice.spkpos(
                target, et, "MOON_ME", "NONE", "MOON"
            )
            site_to_target = np.asarray(target_from_moon, dtype=float) - observer_xyz_km
            az_deg, el_deg = local_angles(site_to_target)
            records.append({
                "point_id": POINT_ID,
                "utc": epoch_utc.isoformat().replace("+00:00", "Z"),
                "et_seconds_past_j2000_tdb": et,
                "target": target,
                "azimuth_deg_clockwise_from_north": az_deg,
                "geometric_center_elevation_deg": el_deg,
                "range_km": float(np.linalg.norm(site_to_target)),
                "aberration_correction": "NONE",
                "reference_frame": "MOON_ME",
                "terrain_visibility_calculated": False,
                "disk_fraction_calculated": False,
            })
        if hour_idx == 1 or hour_idx % 6 == 0 or hour_idx == n_epochs:
            log(f"  ephemeris [{len(records)}/{n_total}] through {epoch_utc.strftime('%H:%M')}Z")

    results = pd.DataFrame(records)
    assert len(results) == 48
    assert results["geometric_center_elevation_deg"].between(-90, 90).all()
    display(results.head(8))

In [ ]:
with run_step("write CSV and run manifest"):
    csv_path = OUTPUT_DIR / "S04-P02_2025-01-01_hourly_geometric_angles.csv"
    results.to_csv(csv_path, index=False)

    run_manifest = {
        "run_utc": datetime.now(timezone.utc).isoformat(),
        "status": "geometric baseline; not terrain visibility",
        "product": "NASA PGDA Product 78, Site04 / Shackleton rim",
        "product_url": "https://pgda.gsfc.nasa.gov/products/78",
        "paper_url": "https://ntrs.nasa.gov/citations/20205009660",
        "point": {
            "id": POINT_ID,
            "x_m": X_M,
            "y_m": Y_M,
            "row": int(row),
            "col": int(col),
            "z_m_above_reference_sphere": z_sample,
            "ldec": ldec_sample,
            "longitude_deg_from_raster_crs": longitude_deg,
            "latitude_deg_from_raster_crs": latitude_deg,
            "observer_radius_m": observer_radius_m,
        },
        "time_range_utc": [epochs[0].isoformat(), epochs[-1].isoformat()],
        "sampling": "24 hourly UTC epochs, 2025-01-01T00:00:00Z through 23:00:00Z",
        "reference_frame": "MOON_ME",
        "ephemeris": "DE421",
        "aberration_correction": "NONE",
        "local_vertical": "radial normal to the 1,737,400 m reference sphere",
        "terrain_visibility_calculated": False,
        "extended_target_disk_visibility_calculated": False,
        "kernel_manifest": kernel_manifest,
    }
    manifest_path = OUTPUT_DIR / "kernel_and_run_manifest.json"
    manifest_path.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
    print("saved_csv:", csv_path)
    print("saved_manifest:", manifest_path)
    print("rows:", len(results))

log(f"notebook finished — total wall time {time.perf_counter() - NOTEBOOK_RUN_T0:.1f}s")
print("SHA-256 values identify the files used; they do not constitute independent validation.")


## 4. Review checklist before treating this as a benchmark result

- Confirm all five downloaded kernel files and their hashes are present in the saved manifest.
- Confirm 48 rows exist: 24 epochs × Sun/Earth.
- Inspect the projection round-trip residual, latitude/longitude, point radius, row/column, Z, and LDEC.
- Compare ellipsoid-normal angles against NAIF `azlcpo_c` in a separate validation step before claiming B4 passed. This notebook's radial spherical normal is not the ellipsoid-normal comparison.
- Compare any future terrain-aware Sun illumination against a matching published product only after matching its ROI, sensor height, date interval, sampling, and disk-visibility definition.
- Keep `SCIENCE_CALCULATIONS_READY=false`. These outputs are a first geometric baseline, not terrain-cleared visibility or validated site recommendations.

### References

- PGDA Product 78 and data description: https://pgda.gsfc.nasa.gov/products/78
- Barker et al. (2021), NASA NTRS copy: https://ntrs.nasa.gov/citations/20205009660
- NAIF lunar PCK/FK tutorial: https://naif.jpl.nasa.gov/pub/naif/toolkit_docs/Tutorials/pdf/individual_docs/23_lunar-earth_pck-fk.pdf
- NAIF generic kernel directory: https://naif.jpl.nasa.gov/pub/naif/generic_kernels/
